In [ ]:
from nba_api.stats.endpoints import leaguegamelog as log, boxscoretraditionalv2 as trd, boxscoreadvancedv2 as adv, boxscorefourfactorsv2 as ffs, boxscoremiscv2 as msc, boxscoreplayertrackv2 as trk, boxscorescoringv2 as scr#, boxscoreusagev2 as usg
import pandas as pd
import time

# Fetch all game IDs since 2015
import pandas as pd
seasons = range(2015, 2026)
games = pd.DataFrame()
gamelog = pd.DataFrame()

for season in seasons:
    gamelog = log.LeagueGameLog(season=season).get_data_frames()[0][['GAME_ID', 'GAME_DATE', 'TEAM_NAME', 'MATCHUP', 'WL']]
    games = pd.concat([games, gamelog])


In [19]:
import time
from functools import wraps
from requests.exceptions import ReadTimeout, ConnectionError

def retry(retries=3, delay=10):
    """
    Decorator to retry API calls with a specified number of retries and delay between attempts.
    
    Args:
        retries (int): Number of retry attempts
        delay (int): Delay between retries in seconds
    """
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            attempts = 0
            while attempts < retries:
                try:
                    return func(*args, **kwargs)
                except (ReadTimeout, ConnectionError) as e:
                    print(f"Attempt {attempts + 1} failed: {str(e)}")
                    if attempts < retries - 1:
                        print(f"Waiting {delay} seconds before retrying...")
                        time.sleep(delay)
                    attempts += 1
            raise Exception(f"Failed after {retries} attempts")
        return wrapper
    return decorator

In [ ]:
# Fetch boxscore data for each game
def fetch_game_data(gid):
    endpoints = {
        'trd': trd.BoxScoreTraditionalV2,
        'adv': adv.BoxScoreAdvancedV2,
        'ffs': ffs.BoxScoreFourFactorsV2,
        'msc': msc.BoxScoreMiscV2,
        'trk': trk.BoxScorePlayerTrackV2,
        'scr': scr.BoxScoreScoringV2

    }
    game_data = {}
    for key, endpoint in endpoints.items():
        bs = endpoint(game_id=gid)
        get_stats = bs.get_data_frames()[1]  # Team stats are in the second DataFrame
        
        game_data[key] = team_stats
    return game_data

# Combine all data into a single DataFrame
all_data = []
data = []
for gid in games['GAME_ID']:
    data = fetch_game_data(gid)
    merged = pd.concat([df.set_index('TEAM_ID') for df in data.values()], axis=1).reset_index()
    merged['GAME_ID'] = gid
    all_data.append(merged)
#    time.sleep(1)

final_df = pd.concat(all_data, ignore_index=True)

ReadTimeout: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)

In [ ]:
from nba_api.stats.endpoints import leaguegamelog as log, boxscoretraditionalv2 as trd, boxscoreadvancedv2 as adv, boxscorefourfactorsv2 as ffs, boxscoremiscv2 as msc, boxscoreplayertrackv2 as trk, boxscorescoringv2 as scr, boxscoreusagev2 as usg
trd.BoxScoreTraditionalV2(game_id=gid)

TypeError: BoxScoreTraditionalV2.__init__() missing 1 required positional argument: 'game_id'

In [106]:
trd.BoxScoreTraditionalV2(game_id='0021500003').get_data_frames()[1].iloc[0] # Home team
trd.BoxScoreTraditionalV2(game_id='0021500003').get_data_frames()[1].iloc[1] # Away team

GAME_ID                 0021500003
TEAM_ID                 1610612744
TEAM_NAME                 Warriors
TEAM_ABBREVIATION              GSW
TEAM_CITY             Golden State
MIN                  240.000000:00
FGM                             41
FGA                             96
FG_PCT                       0.427
FG3M                             9
FG3A                            30
FG3_PCT                        0.3
FTM                             20
FTA                             22
FT_PCT                       0.909
OREB                            21
DREB                            35
REB                             56
AST                             29
STL                              8
BLK                              7
TO                              20
PF                              29
PTS                            111
PLUS_MINUS                    16.0
Name: 1, dtype: object

In [70]:
gamelog

[]

In [ ]:
import pandas as pd
from nba_api.stats.endpoints import leaguegamelog as log, boxscoretraditionalv2 as trd, boxscoreadvancedv2 as adv, boxscorefourfactorsv2 as ffs, boxscoremiscv2 as msc, boxscoreplayertrackv2 as trk, boxscorescoringv2 as scr, boxscoreusagev2 as usg

def fetch_and_merge_game_data(game_id):
    """
    Fetches data from multiple APIs for a given game_id, 
    extracts relevant features for each team, 
    and merges them into a single DataFrame.

    Args:
        game_id (str): The unique identifier for the game.

    Returns:
        pandas.DataFrame: A DataFrame containing the merged data for both teams.
    """

    endpoints = {
        'trd': trd.BoxScoreTraditionalV2,
        'adv': adv.BoxScoreAdvancedV2,
        'ffs': ffs.BoxScoreFourFactorsV2,
        'msc': msc.BoxScoreMiscV2,
        'trk': trk.BoxScorePlayerTrackV2,
        'scr': scr.BoxScoreScoringV2,
        'usg': usg.BoxScoreUsageV2
    }

    all_data = []

    for i in range(2):
        for endpoint_name, endpoint_class in endpoints.items():
            try:
                api_response = endpoint_class(game_id=game_id).get_data_frames()
                if len(api_response) > 0:
                    # Access the specific row using team_id 
                    team_data[endpoint_name] = api_response[0].loc[api_response[0]['TEAM_ID'] == team_id] 
                else:
                    team_data[endpoint_name] = None 
            except Exception as e:
                print(f"Error fetching data from {endpoint_name} for {game_id}-{team_id}: {e}")
                team_data[endpoint_name] = None

        all_data.append(pd.DataFrame(team_data))

    merged_df = pd.concat(all_data, ignore_index=True) 

    return merged_df

# Example usage:
game_id = '0021500003'  # Replace with your actual game_id
game_data = fetch_and_merge_game_data(game_id)
print(game_data) 

NameError: name 'team_id' is not defined

In [112]:
scr.BoxScoreScoringV2(game_id='0021500003').get_data_frames()[1]

,GAME_ID,TEAM_ID,TEAM_NAME,TEAM_ABBREVIATION,TEAM_CITY,MIN,PCT_FGA_2PT,PCT_FGA_3PT,PCT_PTS_2PT,PCT_PTS_2PT_MR,...,PCT_PTS_FB,PCT_PTS_FT,PCT_PTS_OFF_TOV,PCT_PTS_PAINT,PCT_AST_2PM,PCT_UAST_2PM,PCT_AST_3PM,PCT_UAST_3PM,PCT_AST_FGM,PCT_UAST_FGM
0,0021500003,1610612740,Pelicans,NOP,New Orleans,240.000000:00,0.783,0.217,0.611,0.189,...,0.189,0.20,0.305,0.421,0.586,0.414,0.667,0.333,0.600,0.400
1,0021500003,1610612744,Warriors,GSW,Golden State,240.000000:00,0.688,0.313,0.577,0.090,...,0.117,0.18,0.234,0.486,0.688,0.313,0.778,0.222,0.707,0.293


In [138]:
games[(games.GAME_ID=='0021501190') & (games.TEAM_NAME=='Golden State Warriors')]

AttributeError: 'DataFrame' object has no attribute 'GAME_ID'

In [162]:
from nba_api.stats.endpoints import leaguegamelog as log, boxscoretraditionalv2 as trd, boxscoreadvancedv2 as adv, boxscorefourfactorsv2 as ffs, boxscoremiscv2 as msc, boxscoreplayertrackv2 as trk, boxscorescoringv2 as scr
import pandas as pd
import time

def fetch_game_stats(df, endpoints):
    """
    Fetches game statistics from multiple API endpoints and merges them into the game log DataFrame.

    Parameters:
    - games_df: DataFrame containing game log information.
    - endpoints: Dictionary of API endpoints for fetching game statistics.

    Returns:
    - DataFrame with merged game statistics.
    """
    all_stats = pd.DataFrame()

    for index, game in df.iterrows():
        game_id = game['GAME_ID']
        for key, endpoint in endpoints.items():
            try:
                # Fetch data for each endpoint
                stats = endpoint(game_id=game_id).get_data_frames()[0]
                # Add a column to identify the endpoint
                stats['SOURCE'] = key
                # Merge stats with the game log
                all_stats = pd.concat([all_stats, stats])
            except Exception as e:
                print(f"Error fetching data for game {game_id} from {key}: {e}")
            time.sleep(1)  # To avoid hitting rate limits

    # Merge all stats with the original games DataFrame
    merged_df = pd.merge(df, all_stats, on=['GAME_ID', 'TEAM_ID'], how='left')
    return merged_df

endpoints = {
#    'trd': trd.BoxScoreTraditionalV2,
    'adv': adv.BoxScoreAdvancedV2,
#    'ffs': ffs.BoxScoreFourFactorsV2,
#    'msc': msc.BoxScoreMiscV2,
#    'trk': trk.BoxScorePlayerTrackV2,
#    'scr': scr.BoxScoreScoringV2
}

merged_games = fetch_game_stats(games_2025, endpoints)

/var/folders/dg/1nl94b9j41j072s2yp20q5n80000gn/T/ipykernel_13449/1501937873.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_stats = pd.concat([all_stats, stats])
/var/folders/dg/1nl94b9j41j072s2yp20q5n80000gn/T/ipykernel_13449/1501937873.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_stats = pd.concat([all_stats, stats])


KeyError: 'TEAM_ID'

In [158]:
games_2025 = log.LeagueGameLog(season='2024').get_data_frames()[0][['GAME_ID', 'GAME_DATE', 'TEAM_NAME', 'MATCHUP', 'WL']]

In [ ]:
merged_games

,GAME_ID,GAME_DATE,TEAM_NAME,MATCHUP,WL
0,0022400062,2024-10-22,Los Angeles Lakers,LAL vs. MIN,W
1,0022400061,2024-10-22,New York Knicks,NYK @ BOS,L
2,0022400062,2024-10-22,Minnesota Timberwolves,MIN @ LAL,L
3,0022400061,2024-10-22,Boston Celtics,BOS vs. NYK,W
4,0022400066,2024-10-23,Milwaukee Bucks,MIL @ PHI,W
...,...,...,...,...,...
1333,0022400644,2025-01-25,Golden State Warriors,GSW vs. LAL,None
1334,0022400644,2025-01-25,Los Angeles Lakers,LAL @ GSW,None
1335,0022400639,2025-01-25,Toronto Raptors,TOR @ ATL,None
1336,0022400639,2025-01-25,Atlanta Hawks,ATL vs. TOR,None


In [2]:
from nba_api.stats.endpoints import leaguegamelog as log, boxscoretraditionalv2 as trd, boxscoreadvancedv2 as adv, boxscorefourfactorsv2 as ffs, boxscoremiscv2 as msc, boxscoreplayertrackv2 as trk, boxscorescoringv2 as scr#, boxscoreusagev2 as usg
games_2025 = log.LeagueGameLog(season='2024').get_data_frames()[0]
games_2025

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE
0,22024,1610612750,MIN,Minnesota Timberwolves,0022400062,2024-10-22,MIN @ LAL,L,240,35,...,35,47,17,4,1,16,22,103,-7,1
1,22024,1610612747,LAL,Los Angeles Lakers,0022400062,2024-10-22,LAL vs. MIN,W,240,42,...,31,46,22,7,8,7,22,110,7,1
2,22024,1610612752,NYK,New York Knicks,0022400061,2024-10-22,NYK @ BOS,L,240,43,...,29,34,20,2,3,12,12,109,-23,1
3,22024,1610612738,BOS,Boston Celtics,0022400061,2024-10-22,BOS vs. NYK,W,240,48,...,29,40,33,6,3,4,15,132,23,1
4,22024,1610612746,LAC,LA Clippers,0022400071,2024-10-23,LAC vs. PHX,L,265,42,...,38,51,27,9,1,22,29,113,-3,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1367,22024,1610612748,MIA,Miami Heat,0022400652,2025-01-27,MIA vs. ORL,W,290,48,...,38,43,34,9,2,18,21,125,6,1
1368,22024,1610612747,LAL,Los Angeles Lakers,0022400660,2025-01-28,LAL @ PHI,None,120,25,...,18,21,21,2,0,12,13,67,-13,0
1369,22024,1610612755,PHI,Philadelphia 76ers,0022400660,2025-01-28,PHI vs. LAL,None,120,27,...,15,19,13,7,2,4,8,80,13,0
1370,22024,1610612737,ATL,Atlanta Hawks,0022400532,2025-01-28,ATL vs. HOU,None,120,23,...,20,26,12,8,3,13,7,57,-1,0


In [ ]:
games_2025.

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE
0,22024,1610612750,MIN,Minnesota Timberwolves,0022400062,2024-10-22,MIN @ LAL,L,240,35,...,35,47,17,4,1,16,22,103,-7,1
1,22024,1610612747,LAL,Los Angeles Lakers,0022400062,2024-10-22,LAL vs. MIN,W,240,42,...,31,46,22,7,8,7,22,110,7,1
2,22024,1610612752,NYK,New York Knicks,0022400061,2024-10-22,NYK @ BOS,L,240,43,...,29,34,20,2,3,12,12,109,-23,1
3,22024,1610612738,BOS,Boston Celtics,0022400061,2024-10-22,BOS vs. NYK,W,240,48,...,29,40,33,6,3,4,15,132,23,1
4,22024,1610612746,LAC,LA Clippers,0022400071,2024-10-23,LAC vs. PHX,L,265,42,...,38,51,27,9,1,22,29,113,-3,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1367,22024,1610612748,MIA,Miami Heat,0022400652,2025-01-27,MIA vs. ORL,W,290,48,...,38,43,34,9,2,18,21,125,6,1
1368,22024,1610612747,LAL,Los Angeles Lakers,0022400660,2025-01-28,LAL @ PHI,None,120,25,...,18,21,21,2,0,12,13,67,-13,0
1369,22024,1610612755,PHI,Philadelphia 76ers,0022400660,2025-01-28,PHI vs. LAL,None,120,27,...,15,19,13,7,2,4,8,80,13,0
1370,22024,1610612737,ATL,Atlanta Hawks,0022400532,2025-01-28,ATL vs. HOU,None,120,23,...,20,26,12,8,3,13,7,57,-1,0


In [20]:

import pandas as pd
from retry_decorator import retry

@retry(retries=3, delay=10)
def fetch_game_data(game_id):
    trd_temp = trd.BoxScoreTraditionalV2(game_id=game_id, timeout=180).get_data_frames()[1]
    return trd_temp

trd_stats = pd.DataFrame()
for game in games_2025.GAME_ID:
    try:
        game_data = fetch_game_data(game)
        trd_stats = pd.concat([trd_stats, game_data], ignore_index=True)
    except Exception as e:
        print(f"Failed to fetch data for game {game}: {str(e)}")
        continue

ModuleNotFoundError: No module named 'retry_decorator'